In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")  # adjust if your notebook isn't inside notebooks/

orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

print("Loaded all 9 tables")

Loaded all 9 tables


In [2]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 8)
order_items: (112650, 7)
customers: (99441, 5)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


In [3]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [4]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [5]:
print("Unique customer_id:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())

Unique customer_id: 99441
Unique customer_unique_id: 96096


In [6]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [7]:
delivered_orders = orders[orders["order_status"] == "delivered"].copy()
print(f"Filtered from {len(orders)} to {len(delivered_orders)} delivered orders")

Filtered from 99441 to 96478 delivered orders


In [11]:
order_items_agg = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    avg_item_price=("price", "mean"),
).reset_index()

order_items_agg.head(10)

,order_id,n_items,total_price,total_freight,avg_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,21.90,12.69,21.90
6,00054e8431b9d7675808bcb819fb4a32,1,19.90,11.85,19.90
7,000576fe39319847cbb9d288c5617fa6,1,810.00,70.75,810.00
8,0005a1a1728c9d785b8e2b08b904576c,1,145.95,11.65,145.95
9,0005f50442cb953dcd1d21e1fb923495,1,53.99,11.40,53.99


In [13]:
payments_agg = payments.groupby("order_id").agg(
    n_payment_methods=("payment_type", "nunique"),
    total_payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
).reset_index()

# capture the most common/primary payment type separately
primary_payment_type = (
    payments.sort_values("payment_value", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")[["order_id", "payment_type"]]
    .rename(columns={"payment_type": "primary_payment_type"})
)

payments_agg = payments_agg.merge(primary_payment_type, on="order_id", how="left")
payments_agg.head(10)

,order_id,n_payment_methods,total_payment_value,max_installments,primary_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3,credit_card
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,34.59,1,boleto
6,00054e8431b9d7675808bcb819fb4a32,1,31.75,1,credit_card
7,000576fe39319847cbb9d288c5617fa6,1,880.75,10,credit_card
8,0005a1a1728c9d785b8e2b08b904576c,1,157.60,3,credit_card
9,0005f50442cb953dcd1d21e1fb923495,1,65.39,1,credit_card


In [18]:
reviews_agg = (
    reviews.sort_values("review_creation_date", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
    [["order_id", "review_score"]]
    .reset_index(drop = True)
)

reviews_agg.head(10)

,order_id,review_score
0,4d4ef6cd8147028ffd8c18dc0f6e3050,5
1,14a05c02d483800864e5c19a3a7f0ee2,2
2,25be8e0ff01817a8a96f61605da2e447,4
3,4cae9d93ca1f1266e9ddfc79f7ff85db,4
4,f6c9fe3ff737f5568e352b5b2afcf112,5
5,734fd23f82c457088580cf5a8c9da909,5
6,3064071cf67a2cc381cd53b13055eac5,5
7,476b812a7e4fc972646eb390517bddcb,5
8,0128412231f9fe9d7e944eea5392fc6b,5
9,8624511aa767a83ed91a47c158320964,1


In [19]:
order_level = (
    delivered_orders
    .merge(order_items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]], on="customer_id", how="left")
)

print(order_level.shape)
order_level.head()

(96478, 21)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,total_price,...,avg_item_price,n_payment_methods,total_payment_value,max_installments,primary_payment_type,review_score,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,29.99,...,29.99,2.0,38.71,1.0,voucher,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,118.70,...,118.70,1.0,141.46,1.0,boleto,4.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,159.90,...,159.90,1.0,179.12,3.0,credit_card,5.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,45.00,...,45.00,1.0,72.20,1.0,credit_card,5.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,19.90,...,19.90,1.0,28.62,1.0,credit_card,5.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [20]:
order_level.isna().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
order_delivered_carrier_date       2
order_delivered_customer_date      8
order_estimated_delivery_date      0
n_items                            0
total_price                        0
total_freight                      0
avg_item_price                     0
n_payment_methods                  1
total_payment_value                1
max_installments                   1
primary_payment_type               1
review_score                     646
customer_unique_id                 0
customer_zip_code_prefix           0
customer_city                      0
customer_state                     0
dtype: int64

In [21]:
order_level[order_level["total_payment_value"].isna()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,total_price,...,avg_item_price,n_payment_methods,total_payment_value,max_installments,primary_payment_type,review_score,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
29811,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,3,134.97,...,44.99,NaN,NaN,NaN,NaN,1.0,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP


In [22]:
from pathlib import Path

INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

order_level.to_parquet(INTERIM_DIR / "order_level.parquet", index=False)
print(f"Saved {order_level.shape} to {INTERIM_DIR / 'order_level.parquet'}")

Saved (96478, 21) to ../data/interim/order_level.parquet


In [23]:
customers.to_parquet(INTERIM_DIR / "customers.parquet", index=False)
geolocation.to_parquet(INTERIM_DIR / "geolocation.parquet", index=False)